# 01. 基本 CRUD（Spark）

Iceberg テーブルを作り、INSERT / UPDATE / DELETE / MERGE を実行します。
普通の SQL と同じように書けること、そして **書き込みのたびにスナップショットが増えていく** ことを確認します。

- カタログ: `lakehouse`（Polaris）
- 名前空間: `handson`
- テーブル: `handson.crud_spark`

## 準備: SparkSession を作る

カタログの接続設定（Polaris の URL、認証情報など）は `spark-defaults.conf` に書いてあるので、ここでは何も指定しません。
既定のカタログは `lakehouse` なので、`handson.crud_spark` と書けば `lakehouse.handson.crud_spark` を指します。

In [ ]:
from decimal import Decimal
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("01_basic_crud").getOrCreate()

def sql(query):
    """SQL を実行し、結果があれば表示する"""
    df = spark.sql(query)
    if df.columns:
        df.show(truncate=False)

## 1. テーブルを作る

`USING iceberg` を付けると Iceberg テーブルになります。
何度でも最初からやり直せるよう、先に `DROP TABLE ... PURGE`（データファイルごと削除）しておきます。

In [ ]:
sql("DROP TABLE IF EXISTS handson.crud_spark PURGE")
sql("""
CREATE TABLE handson.crud_spark (
    trip_id    BIGINT,
    vendor     STRING,
    fare       DECIMAL(10, 2),
    pickup_at  TIMESTAMP
) USING iceberg
""")
sql("DESCRIBE TABLE EXTENDED handson.crud_spark")

`DESCRIBE TABLE EXTENDED` の `Location` が、このテーブルの置き場所（RustFS 上の S3 パス）です。
RustFS のコンソール（http://localhost:9001 ）で `warehouse` バケットを開くと、`metadata/` フォルダだけができているはずです。

## 2. INSERT

In [ ]:
sql("""
INSERT INTO handson.crud_spark VALUES
    (1, 'A', 12.5, TIMESTAMP '2025-01-01 08:00:00'),
    (2, 'B', 30.0, TIMESTAMP '2025-01-01 09:15:00'),
    (3, 'A',  8.0, TIMESTAMP '2025-01-02 18:30:00')
""")
sql("SELECT * FROM handson.crud_spark ORDER BY trip_id")

コンソールを見ると、今度は `data/` フォルダに Parquet ファイルができています。

## 3. UPDATE

Iceberg は既存のファイルを書き換えません。
更新対象を含むファイルを **新しいファイルとして書き直し**、新しいスナップショットで古いファイルと差し替えます（copy-on-write）。

In [ ]:
sql("UPDATE handson.crud_spark SET fare = fare * 1.1 WHERE vendor = 'A'")
sql("SELECT * FROM handson.crud_spark ORDER BY trip_id")

## 4. DELETE

In [ ]:
sql("DELETE FROM handson.crud_spark WHERE trip_id = 2")
sql("SELECT * FROM handson.crud_spark ORDER BY trip_id")

## 5. MERGE

「あれば更新、なければ追加」を1文で書けます（upsert）。
更新内容を一時ビューとして用意し、`trip_id` で突き合わせます。

In [ ]:
spark.createDataFrame(
    [(1, "A", Decimal("15.00"), None), (4, "C", Decimal("22.00"), None)],
    "trip_id BIGINT, vendor STRING, fare DECIMAL(10,2), pickup_at TIMESTAMP",
).createOrReplaceTempView("updates")

sql("""
MERGE INTO handson.crud_spark AS t
USING updates AS u
ON t.trip_id = u.trip_id
WHEN MATCHED THEN UPDATE SET t.fare = u.fare
WHEN NOT MATCHED THEN INSERT (trip_id, vendor, fare, pickup_at)
                      VALUES (u.trip_id, u.vendor, u.fare, current_timestamp())
""")
sql("SELECT * FROM handson.crud_spark ORDER BY trip_id")

## 6. スナップショットを覗いてみる

ここまでの書き込み（INSERT、UPDATE、DELETE、MERGE）は、それぞれ1つのスナップショットになっています。
`<テーブル名>.snapshots` はメタデータテーブルで、08 で詳しく扱います。次の 02 では、これらのスナップショットを使ってタイムトラベルします。

In [ ]:
sql("SELECT committed_at, snapshot_id, operation FROM handson.crud_spark.snapshots ORDER BY committed_at")

## まとめ

- Iceberg テーブルは `USING iceberg` を付けるだけで作れ、普通の SQL で CRUD できる
- 書き込みのたびに新しいファイルとスナップショットが作られ、既存のファイルは書き換えられない
- テーブルは消さずに残しておきます。Trino からも `SELECT * FROM lakehouse.handson.crud_spark` で読めます（05 で詳しく扱います）